# 63 - Cross-Dataset Evaluation: Secondary → Primer Test

**Skema 2:** Train di dataset sekunder, evaluasi di data test Primer.

| Train | Test | Catatan |
|-------|------|---------|
| CK+ (636/654) | Primer test (929) | Subject-wise 90/10 train/val split |
| JAFFE (213) | Primer test | Subject-wise 90/10 |
| RAF-DB (11,565) | Primer test | Official train, 90/10 untuk val |
| KDEF (2,630) | Primer test | Train set KDEF (sudah ada val terpisah) |

**Model:** Semua model (CNN, FCNN, Intermediate, CNN_TL, Intermediate_TL, Late_Fusion).
**Metrik:** Macro/Micro/Weighted F1.
**Catatan label:** Semua dataset pakai mapping 7-kelas yang sama → aman untuk cross-evaluate.

In [1]:
import sys, os, json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from collections import Counter
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, accuracy_score
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from training.models import (
    EmotionCNN, EmotionFCNN, IntermediateFusion,
    EmotionCNNTransfer, IntermediateFusionTransfer,
)
from training.utils import train_model, full_evaluation

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

BATCH_SIZE = 16
EPOCHS = 40
PATIENCE = 12
OUTPUT_DIR = PROJECT_ROOT / 'models' / 'benchmark' / 'crossdataset'
os.makedirs(OUTPUT_DIR, exist_ok=True)

BENCHMARK_DIR = PROJECT_ROOT / 'data' / 'benchmark'
PRIMER_DIR = PROJECT_ROOT / 'data' / 'dataset_frontonly_conf60'

MODELS = [
    ('CNN', EmotionCNN, 'cnn', 0.0001),
    ('FCNN', EmotionFCNN, 'fcnn', 0.0001),
    ('Intermediate', IntermediateFusion, 'fusion', 0.0001),
    ('CNN_TL', EmotionCNNTransfer, 'cnn', 0.00005),
    ('Intermediate_TL', IntermediateFusionTransfer, 'fusion', 0.00005),
]

EMOTIONS_7 = ['neutral', 'happy', 'sad', 'angry', 'fearful', 'disgusted', 'surprised']
EMOTIONS_4 = ['neutral', 'happy', 'sad', 'negative']
REMAP_4 = np.array([0, 1, 2, 3, 3, 3, 3], dtype=np.int64)

print('Setup complete.')

Device: cuda
Setup complete.


In [2]:
# ── Helpers ──

def make_loader(images, landmarks, labels, model_type, batch_size=32, shuffle=True):
    img_t = torch.from_numpy(images).permute(0, 3, 1, 2)
    lm_t = torch.from_numpy(landmarks)
    y_t = torch.from_numpy(labels).long()
    if model_type == 'cnn': ds = TensorDataset(img_t, y_t)
    elif model_type == 'fcnn': ds = TensorDataset(lm_t, y_t)
    else: ds = TensorDataset(img_t, lm_t, y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, pin_memory=True, drop_last=shuffle)


def metrics_triple(y_true, y_pred):
    return {
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'micro_f1': float(f1_score(y_true, y_pred, average='micro', zero_division=0)),
        'weighted_f1': float(f1_score(y_true, y_pred, average='weighted', zero_division=0)),
        'accuracy': float(accuracy_score(y_true, y_pred)),
    }


def load_primer_test(num_classes):
    te_img = np.load(PRIMER_DIR / 'X_test_images.npy')
    te_lm = np.load(PRIMER_DIR / 'X_test_landmarks.npy')
    y = np.load(PRIMER_DIR / 'y_test.npy')
    if num_classes == 4:
        y = REMAP_4[y]
    return te_img, te_lm, y


def load_secondary(dataset_name, num_classes):
    """Return tr_img, tr_lm, tr_y, v_img, v_lm, v_y (all 7 or 4 class)."""
    if dataset_name == 'ckplus':
        sub = f'ckplus_{num_classes}class' if num_classes == 7 else 'ckplus_4class_contempt'
        d = BENCHMARK_DIR / sub
        X = np.load(d / 'X_images.npy')
        L = np.load(d / 'X_landmarks.npy')
        y = np.load(d / 'y_labels.npy')
        subs = np.load(d / 'subjects.npy', allow_pickle=True)
        # Subject-wise 90/10
        uniq = sorted(set(subs.tolist()))
        rng = np.random.RandomState(42); rng.shuffle(uniq)
        n_tr = int(len(uniq) * 0.9)
        tr_subs = set(uniq[:n_tr]); v_subs = set(uniq[n_tr:])
        tr_idx = np.where(np.isin(subs, list(tr_subs)))[0]
        v_idx = np.where(np.isin(subs, list(v_subs)))[0]
        return X[tr_idx], L[tr_idx], y[tr_idx], X[v_idx], L[v_idx], y[v_idx]

    if dataset_name == 'jaffe':
        d = BENCHMARK_DIR / f'jaffe_{num_classes}class'
        X = np.load(d / 'X_images.npy')
        L = np.load(d / 'X_landmarks.npy')
        y = np.load(d / 'y_labels.npy')
        subs = np.load(d / 'subjects.npy', allow_pickle=True)
        uniq = sorted(set(subs.tolist()))
        rng = np.random.RandomState(42); rng.shuffle(uniq)
        n_tr = int(len(uniq) * 0.9)
        tr_subs = set(uniq[:n_tr]); v_subs = set(uniq[n_tr:])
        tr_idx = np.where(np.isin(subs, list(tr_subs)))[0]
        v_idx = np.where(np.isin(subs, list(v_subs)))[0]
        return X[tr_idx], L[tr_idx], y[tr_idx], X[v_idx], L[v_idx], y[v_idx]

    if dataset_name == 'rafdb':
        d = BENCHMARK_DIR / f'rafdb_{num_classes}class'
        X = np.load(d / 'X_train_images.npy')
        L = np.load(d / 'X_train_landmarks.npy')
        y = np.load(d / 'y_train.npy')
        tr_idx, v_idx = train_test_split(np.arange(len(y)), test_size=0.1, stratify=y, random_state=42)
        return X[tr_idx], L[tr_idx], y[tr_idx], X[v_idx], L[v_idx], y[v_idx]

    if dataset_name == 'kdef':
        d = BENCHMARK_DIR / f'kdef_{num_classes}class'
        tr_X = np.load(d / 'X_train_images.npy')
        tr_L = np.load(d / 'X_train_landmarks.npy')
        tr_y = np.load(d / 'y_train.npy')
        v_X = np.load(d / 'X_val_images.npy')
        v_L = np.load(d / 'X_val_landmarks.npy')
        v_y = np.load(d / 'y_val.npy')
        return tr_X, tr_L, tr_y, v_X, v_L, v_y

    raise ValueError(f'Unknown dataset {dataset_name}')


def train_and_eval(ModelClass, model_type, lr,
                   tr_img, tr_lm, tr_y, v_img, v_lm, v_y,
                   te_img, te_lm, te_y, emotions, save_dir):
    crit = nn.CrossEntropyLoss()
    save_path = str(save_dir / 'model.pth')
    test_loader = make_loader(te_img, te_lm, te_y, model_type, BATCH_SIZE, False)
    if (save_dir / 'model.pth').exists():
        print(f'    [SKIP] model.pth exists, loading and evaluating...')
        model = ModelClass(num_classes=len(emotions)).to(device)
        model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
        r = full_evaluation(model, test_loader, crit, device, model_type, emotions)
        return {'accuracy': float(r['test_accuracy']),
                'macro_f1': float(r['test_macro_f1']),
                'micro_f1': float(r['test_micro_f1']),
                'weighted_f1': float(r['test_weighted_f1'])}
    tr_loader = make_loader(tr_img, tr_lm, tr_y, model_type, BATCH_SIZE)
    val_loader = make_loader(v_img, v_lm, v_y, model_type, BATCH_SIZE, False)
    model = ModelClass(num_classes=len(emotions)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=8, min_lr=1e-7)
    train_model(model, tr_loader, val_loader, crit, opt, sch, device, model_type, EPOCHS, PATIENCE, save_path)
    model.load_state_dict(torch.load(save_path, map_location=device, weights_only=True))
    r = full_evaluation(model, test_loader, crit, device, model_type, emotions)
    return {'accuracy': float(r['test_accuracy']),
            'macro_f1': float(r['test_macro_f1']),
            'micro_f1': float(r['test_micro_f1']),
            'weighted_f1': float(r['test_weighted_f1'])}


def late_fusion_eval(tr_img, tr_lm, tr_y, v_img, v_lm, v_y,
                     te_img, te_lm, te_y, num_classes, save_dir):
    cnn = EmotionCNN(num_classes=num_classes).to(device)
    if (save_dir / 'cnn.pth').exists():
        print('    [SKIP] cnn.pth exists')
    else:
        tr_cnn = make_loader(tr_img, tr_lm, tr_y, 'cnn', BATCH_SIZE)
        val_cnn = make_loader(v_img, v_lm, v_y, 'cnn', BATCH_SIZE, False)
        opt = torch.optim.Adam(cnn.parameters(), lr=0.0001)
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=8, min_lr=1e-7)
        train_model(cnn, tr_cnn, val_cnn, nn.CrossEntropyLoss(), opt, sch,
                    device, 'cnn', EPOCHS, PATIENCE, str(save_dir / 'cnn.pth'))
    fcnn = EmotionFCNN(num_classes=num_classes).to(device)
    if (save_dir / 'fcnn.pth').exists():
        print('    [SKIP] fcnn.pth exists')
    else:
        tr_fcnn = make_loader(tr_img, tr_lm, tr_y, 'fcnn', BATCH_SIZE)
        val_fcnn = make_loader(v_img, v_lm, v_y, 'fcnn', BATCH_SIZE, False)
        opt2 = torch.optim.Adam(fcnn.parameters(), lr=0.0001)
        sch2 = torch.optim.lr_scheduler.ReduceLROnPlateau(opt2, mode='max', factor=0.5, patience=8, min_lr=1e-7)
        train_model(fcnn, tr_fcnn, val_fcnn, nn.CrossEntropyLoss(), opt2, sch2,
                    device, 'fcnn', EPOCHS, PATIENCE, str(save_dir / 'fcnn.pth'))
    cnn.load_state_dict(torch.load(save_dir / 'cnn.pth', map_location=device, weights_only=True))
    fcnn.load_state_dict(torch.load(save_dir / 'fcnn.pth', map_location=device, weights_only=True))
    cnn.eval(); fcnn.eval()
    def batched_softmax(model, arr, is_img, bs=32):
        outs = []
        with torch.no_grad():
            for i in range(0, len(arr), bs):
                chunk = arr[i:i+bs]
                if is_img:
                    t = torch.from_numpy(chunk).permute(0, 3, 1, 2).to(device)
                else:
                    t = torch.from_numpy(chunk).to(device)
                p = torch.softmax(model(t), dim=1).cpu().numpy()
                outs.append(p)
                del t
                torch.cuda.empty_cache()
        return np.concatenate(outs, axis=0)

    vc = batched_softmax(cnn, v_img, True)
    vf = batched_softmax(fcnn, v_lm, False)
    best_wf1, best_w = 0, 0.5
    for w in np.arange(0.0, 1.05, 0.05):
        pr = (w * vc + (1-w) * vf).argmax(axis=1)
        f = f1_score(v_y, pr, average='macro', zero_division=0)
        if f > best_wf1: best_wf1 = f; best_w = w
    print(f'    Late-fusion best w(cnn)={best_w:.2f}')
    cp = batched_softmax(cnn, te_img, True)
    fp = batched_softmax(fcnn, te_lm, False)
    preds = (best_w * cp + (1-best_w) * fp).argmax(axis=1)
    m = metrics_triple(te_y, preds)
    m['best_cnn_weight'] = float(best_w)
    return m


def run_cross(dataset_name, num_classes):
    emotions = EMOTIONS_7 if num_classes == 7 else EMOTIONS_4
    print(f"\n{'='*70}")
    print(f'  CROSS: train={dataset_name} ({num_classes}c) -> test=Primer')
    print(f"{'='*70}")
    tr_img, tr_lm, tr_y, v_img, v_lm, v_y = load_secondary(dataset_name, num_classes)
    te_img, te_lm, te_y = load_primer_test(num_classes)
    print(f'  Train={len(tr_y)}  Val={len(v_y)}  PrimerTest={len(te_y)}')
    results = {}
    for model_name, ModelClass, model_type, lr in MODELS + [('Late_Fusion', None, 'late', 0.0001)]:
        key = f'{model_name}_B1'
        print(f'\n  >> {key} ...')
        save_dir = OUTPUT_DIR / f'{dataset_name}_{num_classes}c' / key
        os.makedirs(save_dir, exist_ok=True)
        if model_type == 'late':
            r = late_fusion_eval(tr_img, tr_lm, tr_y, v_img, v_lm, v_y,
                                  te_img, te_lm, te_y, num_classes, save_dir)
        else:
            r = train_and_eval(ModelClass, model_type, lr,
                                tr_img, tr_lm, tr_y, v_img, v_lm, v_y,
                                te_img, te_lm, te_y, emotions, save_dir)
        results[key] = r
        print(f"    Macro={r['macro_f1']:.4f}  Micro={r['micro_f1']:.4f}  Weighted={r['weighted_f1']:.4f}")
    save_path = OUTPUT_DIR / f'cross_{dataset_name}_{num_classes}c.json'
    with open(save_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'\n  Saved: {save_path}')
    return results

print('Helper functions ready.')

Helper functions ready.


## Run Cross-Dataset Experiments

Jalankan per-dataset untuk kontrol waktu (tiap dataset bisa 1-3 jam).

In [3]:
all_cross = {}
# CK+
all_cross['ckplus_7c'] = run_cross('ckplus', 7)
all_cross['ckplus_4c'] = run_cross('ckplus', 4)


  CROSS: train=ckplus (7c) -> test=Primer


  Train=584  Val=52  PrimerTest=929

  >> CNN_B1 ...
    [SKIP] model.pth exists, loading and evaluating...


Test Loss: 1.1580
Test Accuracy: 0.7191
Test Macro F1:    0.1266
Test Micro F1:    0.7191
Test Weighted F1: 0.6349

Classification Report:
              precision    recall  f1-score   support

     neutral       0.75      0.97      0.85       688
       happy       0.18      0.02      0.04       183
         sad       0.00      0.00      0.00        50
       angry       0.00      0.00      0.00         2
     fearful       0.00      0.00      0.00         1
   disgusted       0.00      0.00      0.00         2
   surprised       0.00      0.00      0.00         3

    accuracy                           0.72       929
   macro avg       0.13      0.14      0.13       929
weighted avg       0.59      0.72      0.63       929

    Macro=0.1266  Micro=0.7191  Weighted=0.6349

  >> FCNN_B1 ...
    [SKIP] model.pth exists, loading and evaluating...
Test Loss: 1.1812
Test Accuracy: 0.7729
Test Macro F1:    0.1943
Test Micro F1:    0.7729
Test Weighted F1: 0.7392

Classification Report:
    

Test Loss: 1.6084
Test Accuracy: 0.5361
Test Macro F1:    0.1526
Test Micro F1:    0.5361
Test Weighted F1: 0.5926

Classification Report:
              precision    recall  f1-score   support

     neutral       0.87      0.61      0.72       688
       happy       0.24      0.41      0.31       183
         sad       0.00      0.00      0.00        50
       angry       0.00      0.00      0.00         2
     fearful       0.00      0.00      0.00         1
   disgusted       0.00      0.00      0.00         2
   surprised       0.02      1.00      0.04         3

    accuracy                           0.54       929
   macro avg       0.16      0.29      0.15       929
weighted avg       0.69      0.54      0.59       929

    Macro=0.1526  Micro=0.5361  Weighted=0.5926

  >> CNN_TL_B1 ...
    [SKIP] model.pth exists, loading and evaluating...


Test Loss: 0.9827
Test Accuracy: 0.6695
Test Macro F1:    0.1634
Test Micro F1:    0.6695
Test Weighted F1: 0.7008

Classification Report:
              precision    recall  f1-score   support

     neutral       0.90      0.86      0.88       688
       happy       0.50      0.15      0.23       183
         sad       0.09      0.02      0.03        50
       angry       0.00      0.00      0.00         2
     fearful       0.00      0.00      0.00         1
   disgusted       0.00      0.00      0.00         2
   surprised       0.00      0.00      0.00         3

    accuracy                           0.67       929
   macro avg       0.21      0.15      0.16       929
weighted avg       0.77      0.67      0.70       929

    Macro=0.1634  Micro=0.6695  Weighted=0.7008

  >> Intermediate_TL_B1 ...
    [SKIP] model.pth exists, loading and evaluating...


...

Test Loss: 1.4489
Test Accuracy: 0.3864
Test Macro F1:    0.2579
Test Micro F1:    0.3864
Test Weighted F1: 0.5274

Classification Report:
              precision    recall  f1-score   support

     neutral       0.97      0.45      0.61       688
       happy       0.87      0.22      0.36       183
         sad       0.17      0.02      0.04        50
    negative       0.01      0.88      0.02         8

    accuracy                           0.39       929
   macro avg       0.50      0.39      0.26       929
weighted avg       0.90      0.39      0.53       929

    Macro=0.2579  Micro=0.3864  Weighted=0.5274

  >> Intermediate_TL_B1 ...
    [SKIP] model.pth exists, loading and evaluating...


Test Loss: 1.6773
Test Accuracy: 0.2713
Test Macro F1:    0.2016
Test Micro F1:    0.2713
Test Weighted F1: 0.3811

Classification Report:
              precision    recall  f1-score   support

     neutral       0.98      0.27      0.42       688
       happy       0.39      0.33      0.36       183
         sad       0.00      0.00      0.00        50
    negative       0.01      1.00      0.03         8

    accuracy                           0.27       929
   macro avg       0.35      0.40      0.20       929
weighted avg       0.81      0.27      0.38       929

    Macro=0.2016  Micro=0.2713  Weighted=0.3811

  >> Late_Fusion_B1 ...


    [SKIP] cnn.pth exists
    [SKIP] fcnn.pth exists


    Late-fusion best w(cnn)=0.65


    Macro=0.2448  Micro=0.4887  Weighted=0.5560

  Saved: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/crossdataset/cross_ckplus_4c.json


In [4]:
# JAFFE
all_cross['jaffe_7c'] = run_cross('jaffe', 7)
all_cross['jaffe_4c'] = run_cross('jaffe', 4)


  CROSS: train=jaffe (7c) -> test=Primer


  Train=193  Val=20  PrimerTest=929

  >> CNN_B1 ...


 Epoch  Train Loss  Train Acc   Val Loss   Val Acc   Val F1         LR
---------------------------------------------------------------------------


     1      2.0114     0.1927     1.9631    0.1000   0.0260   0.000100  (1.4s)


     2      1.9503     0.1823     1.9753    0.1500   0.0373   0.000100  (1.3s)


...

    13      1.0943     0.5417     1.1570    0.5000   0.2633   0.000100  (0.1s)
    14      1.0827     0.5729     1.1645    0.5000   0.3406   0.000100  (0.1s)
    15      1.0780     0.5729     1.2132    0.5500   0.3364   0.000100  (0.0s)
    16      1.0106     0.6198     1.1318    0.4000   0.1600   0.000100  (0.1s)


    17      1.0067     0.6094     1.1159    0.4500   0.2381   0.000100  (0.1s)
    18      0.9457     0.6354     1.1491    0.4000   0.1600   0.000100  (0.1s)
    19      0.9396     0.6354     1.1601    0.4500   0.2235   0.000100  (0.1s)
    20      0.9178     0.6354     1.1138    0.4000   0.1667   0.000100  (0.0s)


    21      0.9391     0.6354     1.1370    0.5500   0.3068   0.000050  (0.0s)
    22      0.8955     0.6719     1.1552    0.5500   0.3068   0.000050  (0.0s)
    23      0.8791     0.6771     1.1238    0.5500   0.3068   0.000050  (0.0s)

Early stopping at epoch 23. Best epoch: 11 (val_f1=0.3600)

Best: epoch 11, val_acc=0.5000, val_f1=0.3600
Model saved: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/crossdataset/jaffe_4c/Late_Fusion_B1/fcnn.pth


    Late-fusion best w(cnn)=0.90


    Macro=0.0070  Micro=0.0097  Weighted=0.0023

  Saved: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/crossdataset/cross_jaffe_4c.json


In [5]:
# RAF-DB (paling besar, ~1-3 jam)
all_cross['rafdb_7c'] = run_cross('rafdb', 7)
all_cross['rafdb_4c'] = run_cross('rafdb', 4)


  CROSS: train=rafdb (7c) -> test=Primer


  Train=10408  Val=1157  PrimerTest=929

  >> CNN_B1 ...


 Epoch  Train Loss  Train Acc   Val Loss   Val Acc   Val F1         LR
---------------------------------------------------------------------------


     1      1.5676     0.4190     1.2024    0.5782   0.3524   0.000100  (69.6s)


     2      1.2135     0.5671     0.9534    0.6733   0.4773   0.000100  (70.1s)


...

    38      0.7515     0.6995     0.6976    0.7321   0.6990   0.000050  (2.1s)


    39      0.7556     0.6951     0.6998    0.7243   0.6882   0.000050  (2.1s)


    40      0.7443     0.7024     0.6817    0.7398   0.6905   0.000050  (2.0s)

Best: epoch 38, val_acc=0.7321, val_f1=0.6990
Model saved: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/crossdataset/rafdb_4c/Late_Fusion_B1/fcnn.pth


    Late-fusion best w(cnn)=0.60


    Macro=0.1697  Micro=0.1367  Weighted=0.1466

  Saved: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/crossdataset/cross_rafdb_4c.json


In [6]:
# KDEF
all_cross['kdef_7c'] = run_cross('kdef', 7)
all_cross['kdef_4c'] = run_cross('kdef', 4)


  CROSS: train=kdef (7c) -> test=Primer


  Train=2631  Val=340  PrimerTest=929

  >> CNN_B1 ...


 Epoch  Train Loss  Train Acc   Val Loss   Val Acc   Val F1         LR
---------------------------------------------------------------------------


     1      1.9998     0.1848     1.8998    0.2353   0.1205   0.000100  (17.2s)


     2      1.8678     0.2416     1.7179    0.3500   0.3129   0.000100  (17.3s)


...

    38      0.6326     0.7237     0.6164    0.7059   0.5275   0.000100  (0.5s)


    39      0.6169     0.7363     0.6020    0.7324   0.5875   0.000100  (0.5s)


    40      0.5953     0.7435     0.5942    0.7324   0.6188   0.000100  (0.5s)

Best: epoch 37, val_acc=0.7853, val_f1=0.7296
Model saved: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/crossdataset/kdef_4c/Late_Fusion_B1/fcnn.pth


    Late-fusion best w(cnn)=0.20


    Macro=0.0043  Micro=0.0086  Weighted=0.0001

  Saved: /home/bs000716/MOTHER-TANK/TRAIN/models/benchmark/crossdataset/cross_kdef_4c.json


## Ringkasan Semua Cross-Dataset Results

In [7]:
for key, res in all_cross.items():
    print(f"\n{'='*78}")
    print(f'  {key.upper()} -> Primer Test (sorted by Macro F1)')
    print(f"{'='*78}")
    print(f"  {'Model':<22} {'Macro':>10} {'Micro':>10} {'Weighted':>10} {'Accuracy':>10}")
    print(f"  {'-'*70}")
    for k in sorted(res.keys(), key=lambda k: -res[k]['macro_f1']):
        r = res[k]
        print(f"  {k:<22} {r['macro_f1']:>10.4f} {r['micro_f1']:>10.4f} {r['weighted_f1']:>10.4f} {r['accuracy']:>10.4f}")

# Save all
with open(OUTPUT_DIR / 'all_cross_results.json', 'w') as f:
    json.dump(all_cross, f, indent=2)
print(f'\nAll cross-dataset results saved: {OUTPUT_DIR / "all_cross_results.json"}')


  CKPLUS_7C -> Primer Test (sorted by Macro F1)
  Model                       Macro      Micro   Weighted   Accuracy
  ----------------------------------------------------------------------
  FCNN_B1                    0.1943     0.7729     0.7392     0.7729
  CNN_TL_B1                  0.1634     0.6695     0.7008     0.6695
  Late_Fusion_B1             0.1601     0.5285     0.5801     0.5285
  Intermediate_B1            0.1526     0.5361     0.5926     0.5361
  CNN_B1                     0.1266     0.7191     0.6349     0.7191
  Intermediate_TL_B1         0.1029     0.1518     0.2376     0.1518

  CKPLUS_4C -> Primer Test (sorted by Macro F1)
  Model                       Macro      Micro   Weighted   Accuracy
  ----------------------------------------------------------------------
  CNN_B1                     0.3958     0.6416     0.7031     0.6416
  CNN_TL_B1                  0.2579     0.3864     0.5274     0.3864
  Late_Fusion_B1             0.2448     0.4887     0.5560     0.48